In [ ]:
---
title: "GroupWork_Assignment2"
output: html_document
date: '2023-10-23'
---

```{r setup, include=FALSE}
knitr::opts_chunk$set(echo = TRUE)
```

In [ ]:
```{r Step 1: Loading packages,cache=TRUE,warning=FALSE,message=FALSE,dpi=dpi,out.width="100%"}
# r packages

library(reshape2)
library(ggplot2)
library(cowplot)
library(vegan)
library(tibble)
library(dplyr)
library(DESeq2)
library(scales)
library(cluster)
library(ComplexHeatmap)
# install.packages("ggrepel")
library(ggrepel)
# install.packages("circlize")
library(circlize)
library(plyr)
library("Hmisc")
library(corrplot)
# install.packages("iNEXT")
library(iNEXT)
# install.packages("gridExtra")
library(gridExtra)
```


```{r Step 1: Loading packages,cache=TRUE,warning=FALSE,message=FALSE,dpi=dpi,out.width="100%"}
# Specifying data location
input_path <- "/kyukon/data/gent/courses/2023/ham_I002623/input/AssignmentPartII/"
output_path <- "/data/gent/
L/vsc45876/2023_HAM/GroupWork/Assignment2_Output"

# User define function
source('/kyukon/data/gent/courses/2023/ham_I002623/input/KDP_papertheme_sizeselect.R')
source('/kyukon/data/gent/courses/2023/ham_I002623/input/demo_data analysis/volcanokim.R')
source('/kyukon/data/gent/courses/2023/ham_I002623/input/demo_data analysis/filter_mixomics.R')

# Color vectors
sevencolors = c("#9BD0E3","#DFDA5F","#9FDF9D","#E3B1D2","#E3B573","#6EDFC4","#B2E078")
elevencolors = c("#000000","#CD6600","#36648B","#008B00","#7A378B","#838B83","#8B0000","#323C4D","#A03D44","#98B736","#C8ADA4")
thirtycolors=c("#323F24","#CB51D7","#72E245","#DE4F2D","#81DDC7","#6584C6","#CFAE3C","#914261","#BFDC86","#CDC6BE","#D3469A","#3C315A","#607B30","#DD4469","#5B9072","#CA9FC7","#7F592D","#5A656D","#D1867F","#4C2426","#79B6CE","#6E67D1","#CBDC3F","#63D98B","#97362B","#CBB37D","#7E3F85","#CF8338","#5DA73A","#CB80D0")
fourtycolors = c("#D3469A","#3C315A","#607B30","#DD4469","#5B9072","#CA9FC7","#7F592D","#5A656D","#D1867F","#4C2426","#79B6CE","#6E67D1","#CBDC3F","#63D98B","#97362B","#CBB37D","#7E3F85","#CF8338","#5DA73A","#CB80D0","#7eb96b","#9366e9","#6eb729","#b55edb","#46c353","#a92fa4","#afc52e","#3468e5","#b4a823","#5a4fc4","#82cf62","#e357bc","#4b9925","#cf6bd6","#369039","#e53486","#39c685","#bc2c84","#8eab32","#6b4bb1")
point_colors <- c("#00008B4D","#00008B","#FFA5004D","#FFA500","#000000")
```

In [ ]:
```{r Step 2: Loading data,cache=TRUE,warning=FALSE,message=FALSE,dpi=dpi,out.width="100%"}
# Loading seq data
    G4_ASVtable <- read.csv2(paste(input_path,"ASVtable_partII.csv", sep=""),row.names=1)
    G4_Taxtable <- read.csv2(paste(input_path,"taxtable_partII.csv", sep=""),row.names=1)
    G4_metatable <- read.csv(paste(input_path,"Metadata_partII.csv", sep=""),row.names=1)

# Order factor levels
    G4_metatable$Donor <- factor(G4_metatable$Donor, levels=c("Donor 1","Donor 2", "Donor 3"))
    G4_metatable$Prebiotic <- factor(G4_metatable$Prebiotic, levels=c("no","BRAN"))
    G4_metatable$Antibiotic <- factor(G4_metatable$Antibiotic, levels=c("no", "AB"))
    G4_metatable$Post_Antibiotic <- factor(G4_metatable$Post_Antibiotic, levels=c("no", "AB"))
    G4_metatable$Lumen_Bran <- factor(G4_metatable$Lumen_Bran, levels=c("Lumen", "Bran"))
    G4_metatable$Phase <- factor(G4_metatable$Phase, levels=c("stabilisation", "prebiotic", "AB", "ABwo"))
```

```{r Step 3: Preprocessing data,cache=TRUE,warning=FALSE,message=FALSE,dpi=dpi,out.width="100%"}
 #combining ASV table with taxtable
    G4_ASVtax <- dplyr::select(G4_Taxtable,c(1:6))
    G4_ASVtax <- cbind(G4_ASVtax[colnames(G4_ASVtable),],t(G4_ASVtable))

    G4_ASVtable_prop <- sweep(G4_ASVtable,1,rowSums(G4_ASVtable),'/')
    rowSums(G4_ASVtable_prop)

  ## Aggregate at Genus level
    G4_genusASVtable <- aggregate(. ~ Genus, data=G4_ASVtax[,-c(1:5)], FUN=sum)
    rownames(G4_genusASVtable) <- G4_genusASVtable[,1]
    G4_genusASVtable <- G4_genusASVtable[,-1]

    G4_genusASVtable_prop <- sweep(G4_genusASVtable, 2, colSums(G4_genusASVtable),'/')
    colSums(G4_genusASVtable_prop)

  ## Abundance table at Phylum level and ASV level
    G4_phylumASVtable <- aggregate(. ~ Phylum, data=G4_ASVtax[,-c(1,3:6)], FUN=sum)
    rownames(G4_phylumASVtable) <- G4_phylumASVtable[,1]
    G4_phylumASVtable <- G4_phylumASVtable[,-1]

    G4_phylumASVtable_prop <- sweep(G4_phylumASVtable,2,colSums(G4_phylumASVtable),'/')
    colSums(G4_phylumASVtable_prop)
```

{A.1}

In [ ]:
 ```{r}
  Lumen <- subset(G4_metatable, Lumen_Bran=='Lumen')

# draw timepoint vs. qPCR_av for each donor
  Lumen_qPCR_nona <- Lumen %>% filter(!is.na(qPCR_av))
  Lumen_qPCR_donor1 <- subset(Lumen_qPCR_nona, Donor=='Donor 1')
  Lumen_qPCR_donor2 <- subset(Lumen_qPCR_nona, Donor=='Donor 2')
  Lumen_qPCR_donor3 <- subset(Lumen_qPCR_nona, Donor=='Donor 3')

  p1 <- ggplot(data=Lumen_qPCR_donor1,aes(x=Timepoint, y=qPCR_av)) + geom_line(color = "blue") + ggtitle("Donor1_Lumen_qPCR")
  p2 <- ggplot(data=Lumen_qPCR_donor2,aes(x=Timepoint, y=qPCR_av)) + geom_line(color = "blue") + ggtitle("Donor2_Lumen_qPCR")
  p3 <- ggplot(data=Lumen_qPCR_donor3,aes(x=Timepoint, y=qPCR_av)) + geom_line(color = "blue") + ggtitle("Donor3_Lumen_qPCR")
  # grid.arrange(p1, p2, p3, ncol=1)

       #linegraphs
       plot <- ggplot(Lumen_qPCR_nona,aes(x=Timepoint,y=qPCR_av)) +  geom_pointrange(aes(ymin=Lumen_qPCR_nona$qPCR_av,ymax=Lumen_qPCR_nona$qPCR_av,colour=as.factor(Donor)),position=position_jitter(width=0,height=0.2),size=0.2) + geom_errorbar(aes(ymin = qPCR_av+qPCR_sd, ymax = qPCR_av-qPCR_sd, colour=as.factor(Donor)), width = 0.2)
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("qPCR_av")

       plot


# draw timepoint vs. FCM_av for each donor
  Lumen_FCM_nona <- Lumen %>% filter(!is.na(FCM_av))

  Lumen_FCM_donor1 <- subset(Lumen_FCM_nona, Donor=='Donor 1')
  Lumen_FCM_donor2 <- subset(Lumen_FCM_nona, Donor=='Donor 2')
  Lumen_FCM_donor3 <- subset(Lumen_FCM_nona, Donor=='Donor 3')

  p4 <- ggplot(data=Lumen_FCM_donor1,aes(x=Timepoint, y=FCM_av)) + geom_line(color = "blue") + ggtitle("Donor1_Lumen_FCM")
  p5 <- ggplot(data=Lumen_FCM_donor2,aes(x=Timepoint, y=FCM_av)) + geom_line(color = "blue") + ggtitle("Donor2_Lumen_FCM")
  p6 <- ggplot(data=Lumen_FCM_donor3,aes(x=Timepoint, y=FCM_av)) + geom_line(color = "blue") + ggtitle("Donor3_Lumen_FCM")
  # grid.arrange(p4, p5, p6, ncol=1)

  Lumen_lineGraphs <- grid.arrange(p1, p2, p3, p4, p5, p6, ncol=3)

```

In [ ]:
```{r A.1 Plot the microbial load in the lumen and the microbial biomass concentration attached to the wheat bran particles as a function of time}
   Lumen <- subset(G4_metatable, Lumen_Bran=='Lumen')

# draw timepoint vs. qPCR_av for each donor
  Lumen_qPCR_nona <- Lumen %>% filter(!is.na(qPCR_av))
  Lumen_qPCR_donor1 <- subset(Lumen_qPCR_nona, Donor=='Donor 1')
  Lumen_qPCR_donor2 <- subset(Lumen_qPCR_nona, Donor=='Donor 2')
  Lumen_qPCR_donor3 <- subset(Lumen_qPCR_nona, Donor=='Donor 3')

  p1 <- ggplot(data=Lumen_qPCR_donor1,aes(x=Timepoint, y=qPCR_av)) + geom_line(color = "blue")
  p2 <- ggplot(data=Lumen_qPCR_donor2,aes(x=Timepoint, y=qPCR_av)) + geom_line(color = "blue")
  p3 <- ggplot(data=Lumen_qPCR_donor3,aes(x=Timepoint, y=qPCR_av)) + geom_line(color = "blue")
  grid.arrange(p1, p2, p3, ncol=1)

# draw timepoint vs. FCM_av for each donor
  Lumen_FCM_nona <- Lumen %>% filter(!is.na(FCM_av))
  Lumen_FCM_donor1 <- subset(Lumen_FCM_nona, Donor=='Donor 1')
  Lumen_FCM_donor2 <- subset(Lumen_FCM_nona, Donor=='Donor 2')
  Lumen_FCM_donor3 <- subset(Lumen_FCM_nona, Donor=='Donor 3')

  p4 <- ggplot(data=Lumen_FCM_donor1,aes(x=Timepoint, y=FCM_av)) + geom_line(color = "blue")
  p5 <- ggplot(data=Lumen_FCM_donor2,aes(x=Timepoint, y=FCM_av)) + geom_line(color = "blue")
  p6 <- ggplot(data=Lumen_FCM_donor3,aes(x=Timepoint, y=FCM_av)) + geom_line(color = "blue")
  grid.arrange(p4, p5, p6, ncol=1)

  grid.arrange(p1, p2, p3, p4, p5, p6, ncol=3)

  plots <- lapply(unique(Lumen_FCM_nona$Donor), function(d) {
    data_subset <- subset(Lumen_FCM_nona, Donor == 'Donor 1')
    ggplot(data=Lumen_FCM_donor1,aes(x=Timepoint, y=FCM_av)) + geom_line(color = "blue")})

```


```{r}
# Genus level
    G4_genustable_bar <- decostand(G4_genusASVtable,MARGIN=2,"total")*100
    G4_selecttop20par <- rowSums(G4_genustable_bar)
    G4_genustable_bar <- rownames_to_column(G4_genustable_bar)
    G4_genustable_bar <- as.data.frame(G4_genustable_bar %>% top_n(13, G4_selecttop20par))
    other <- 100-colSums(G4_genustable_bar[,-1])
    rownames(G4_genustable_bar) <- G4_genustable_bar$rowname
    G4_genustable_bar <- G4_genustable_bar[,-1]
    G4_genustable_bar <- rbind(G4_genustable_bar,other)
    rownames(G4_genustable_bar)[nrow(G4_genustable_bar)] <- "Other"

    noNA <- subset(G4_metatable, forward_file!='NA')
    G4_genustable_bar <- cbind(noNA[colnames(G4_genustable_bar),], t(G4_genustable_bar))
    G4_genustable_bar <- melt(G4_genustable_bar,id.vars=colnames(noNA))

    Donor1_genustable_bar <- subset(G4_genustable_bar, Donor == "Donor 1")
    Donor2_genustable_bar <- subset(G4_genustable_bar, Donor == "Donor 2")
    Donor3_genustable_bar <- subset(G4_genustable_bar, Donor == "Donor 3")

    # Donor1
    p <- ggplot(data=Donor1_genustable_bar,aes(x=Timepoint,y=value))
    p <- p + geom_area(aes(fill=variable)) + facet_grid(Donor1_genustable_bar$Lumen_Bran)
    p <- p + theme(axis.text.x = element_text(angle =0),strip.text.y =element_text(angle = 0, hjust = 1))+scale_fill_manual(name='Genus',values=thirtycolors)
    p <- p +scale_x_continuous(breaks=as.numeric(as.character(unique(Donor1_genustable_bar$Timepoint))),labels=unique(Donor1_genustable_bar$Timepoint))
    p <- p + ylab("Relative abundance") + xlab("Days after inoculation") + theme(legend.text=element_text(face='italic')) + guides(fill=guide_legend(ncol=2))
    p
    #ggsave(p, path = "/data/gent/458/vsc45845/2023_HAM/Assignment2_Output/", filename = "Donor1_areagraphs_GenusLevel",
       #device = "png", width = 9, height = 5.5)
```

{A.3}

In [ ]:
```{r A.3 Bargraph trends upon pre- and antibiotic treatment}

# Donor1
p1 <- ggplot(data=Donor1_genustable_bar, aes(x=Lumen_Bran, y=value))
p1 <- p1 +geom_bar(stat="identity",aes(fill=variable), position='fill')
p1 <- papertheme(p1) + facet_grid(.~ Donor1_genustable_bar$Phase, drop=TRUE,scale="free_x",space="free_x") + scale_fill_manual(values=thirtycolors, name="Genus")
p1 <- p1 + ylab("Relative abundance (%)") + xlab("Donor 1") + theme(legend.text=element_text(face='italic'), strip.text.y = element_text(angle=0), legend.position="right") + guides(fill=guide_legend(ncol=1))
p1

# Donor2
p2 <- ggplot(data=Donor2_genustable_bar, aes(x=Lumen_Bran, y=value))
p2 <- p2 +geom_bar(stat="identity",aes(fill=variable), position='fill')
p2 <- papertheme(p2) + facet_grid(.~ Donor2_genustable_bar$Phase, drop=TRUE,scale="free_x",space="free_x") + scale_fill_manual(values=thirtycolors, name="Genus")
p2 <- p2 + ylab("Relative abundance (%)") + xlab("Donor 2") + theme(legend.text=element_text(face='italic'), strip.text.y = element_text(angle=0), legend.position="right") + guides(fill=guide_legend(ncol=1))
p2

# Donor3
p3 <- ggplot(data=Donor3_genustable_bar, aes(x=Lumen_Bran, y=value))
p3 <- p3 +geom_bar(stat="identity",aes(fill=variable), position='fill')
p3 <- papertheme(p3) + facet_grid(.~ Donor3_genustable_bar$Phase, drop=TRUE,scale="free_x",space="free_x") + scale_fill_manual(values=thirtycolors, name="Genus")
p3 <- p3 + ylab("Relative abundance (%)") + xlab("Donor 3") + theme(legend.text=element_text(face='italic'), strip.text.y = element_text(angle=0), legend.position="right") + guides(fill=guide_legend(ncol=1))
p3
```

{B.1}

In [ ]:
```{r}
# Plot the proportional change of metabolites according to the phase
    Metabolites <- G4_metatable[,c(1, 7, 15:24)]
    Metabolites <- na.omit(Metabolites)
    Metabolites <- melt(Metabolites, id.vars=c("Donor", "Phase"))
    Mb1 <- subset(Metabolites,Donor=="Donor 1")
    Mb2 <- subset(Metabolites,Donor=="Donor 2")
    Mb3<- subset(Metabolites,Donor=="Donor 3")

    # Donor1
    mb_d1 <- ggplot(data=Mb1,aes(x=Phase,y=value))
    mb_d1 <- mb_d1 + geom_bar(stat="identity",aes(fill=variable), position='fill')
    mb_d1 <- papertheme(mb_d1) + scale_fill_manual(values=thirtycolors,name=NULL)
    mb_d1 <- mb_d1+ ylab("Donor 1 Metabolites (mM)") + xlab("Phase") + theme(strip.text.y = element_text(angle=0),legend.justification="center") +guides(fill=guide_legend(nrow=2))
    mylegendSCFA<-get_legend(mb_d1)
    mb_d1 <- mb_d1 + theme(legend.position = "none",axis.title.x = element_text(color="white"))

    # Donor 2
    mb_d2 <- ggplot(data=Mb2,aes(x=Phase,y=value))
    mb_d2 <- mb_d2 + geom_bar(stat="identity",aes(fill=variable), position='fill')
    mb_d2 <- papertheme(mb_d2) + scale_fill_manual(values=thirtycolors,name=NULL)
    mb_d2 <- mb_d2+ ylab("Donor 2 Metabolites (mM)") + xlab("Phase") + theme(strip.text.y = element_text(angle=0),legend.justification="center") +guides(fill=guide_legend(nrow=2))
    mb_d2 <- mb_d2 + theme(legend.position = "none",axis.title.x = element_text(color="white"))

    # Donor 3
    mb_d3 <- ggplot(data=Mb3,aes(x=Phase,y=value))
    mb_d3 <- mb_d3 + geom_bar(stat="identity",aes(fill=variable), position='fill')
    mb_d3 <- papertheme(mb_d3) + scale_fill_manual(values=thirtycolors,name=NULL)
    mb_d3 <- mb_d3+ ylab("Donor 3 Metabolites (mM)") + xlab("Phase") + theme(strip.text.y = element_text(angle=0),legend.justification="center") +guides(fill=guide_legend(nrow=2))
    mb_d3 <- mb_d3 + theme(legend.position = "none",axis.title.x = element_text(color="white"))

    dpi <- 300
    tiff("Figure2.tiff",width=5*dpi,height=10*dpi,res=dpi,compression="lzw")
    cowplot::plot_grid(plot_grid(mb_d1,mb_d2,mb_d3,nrow=1,ncol=3,rel_heights=c(0.5,0.5,0.5,0.5),rel_widths=c(0.5,0.5,0.5,0.5),labels='AUTO',label_size=10,label_fontfamily='sans'),mylegendSCFA,nrow=2,rel_heights=c(1,1))
```



In [ ]:
```{r}
    Metabolites <- G4_metatable[,c(1, 7, 15:24)]
    Metabolites <- na.omit(Metabolites)
    Metabolites <- melt(Metabolites, id.vars=c("Donor", "Phase"))
    Mb1 <- subset(Metabolites,Donor=="Donor 1")
    Mb2 <- subset(Metabolites,Donor=="Donor 2")
    Mb3<- subset(Metabolites,Donor=="Donor 3")

    # Donor1
    mb_d1 <- ggplot(data=Mb1,aes(x=Phase,y=value))
    mb_d1 <- mb_d1 + geom_bar(stat="identity",aes(fill=variable))
    mb_d1 <- papertheme(mb_d1) + scale_fill_manual(values=thirtycolors,name=NULL)
    mb_d1 <- mb_d1+ ylab("Donor 1 Metabolites (mM)") + xlab("Phase") + theme(strip.text.y = element_text(angle=0),legend.justification="center") +guides(fill=guide_legend(nrow=2))
    mylegendSCFA <- get_legend(mb_d1)
    mb_d1 <- mb_d1 + theme(legend.position = "none",axis.title.x = element_text(color="white"))

    # Donor 2
    mb_d2 <- ggplot(data=Mb2,aes(x=Phase,y=value))
    mb_d2 <- mb_d2 + geom_bar(stat="identity",aes(fill=variable))
    mb_d2 <- papertheme(mb_d2) + scale_fill_manual(values=thirtycolors,name=NULL)
    mb_d2 <- mb_d2+ ylab("Donor 2 Metabolites (mM)") + xlab("Phase") + theme(strip.text.y = element_text(angle=0),legend.justification="center") +guides(fill=guide_legend(nrow=2))
    mb_d2 <- mb_d2 + theme(legend.position = "none",axis.title.x = element_text(color="white"))

    # Donor 3
    mb_d3 <- ggplot(data=Mb3,aes(x=Phase,y=value))
    mb_d3 <- mb_d3 + geom_bar(stat="identity",aes(fill=variable))
    mb_d3 <- papertheme(mb_d3) + scale_fill_manual(values=thirtycolors,name=NULL)
    mb_d3 <- mb_d3+ ylab("Donor 3 Metabolites (mM)") + xlab("Phase") + theme(strip.text.y = element_text(angle=0),legend.justification="center") +guides(fill=guide_legend(nrow=2))
    mb_d3 <- mb_d3 + theme(legend.position = "none",axis.title.x = element_text(color="white"))

    dpi <- 300
    tiff("Figure2.tiff",width=5*dpi,height=10*dpi,res=dpi,compression="lzw")
    plot <- cowplot::plot_grid(plot_grid(mb_d1,mb_d2,mb_d3,nrow=1,ncol=3,rel_heights=c(0.5,0.5,0.5,0.5),rel_widths=c(0.5,0.5,0.5,0.5),labels='AUTO',label_size=10,label_fontfamily='sans'),mylegendSCFA,nrow=2,rel_heights=c(1,1))

```

In [ ]:
```{r}
  Metabolites <- G4_metatable[,c(1, 7, 15:24)]
    Metabolites <- na.omit(Metabolites)
    Metabolites <- melt(Metabolites, id.vars=c("Donor", "Phase"))
    # Summarize the data to get the sum of 'value' for each 'Phase' and 'variable'
    sum_Metabolites <- Metabolites %>%
    group_by(Donor, Phase, variable) %>%
    dplyr::summarize(total_value = sum(value))
    Mb1 <- subset(sum_Metabolites,Donor=="Donor 1")
    Mb2 <- subset(sum_Metabolites,Donor=="Donor 2")
    Mb3<- subset(sum_Metabolites,Donor=="Donor 3")


    # Donor1
    mb_d1 <- ggplot(data=Mb1, aes(x=Phase, y=total_value, fill=variable)) +
    geom_bar(stat="identity") +
    geom_text(aes(label=ifelse(total_value==0,"", sprintf("%.2f",total_value))), position=position_stack(vjust=0.5),size=3,color="white") +
    ylab("Donor 1 Metabolites (mM)") +
    xlab("Phase") +
    scale_fill_manual(values=thirtycolors, name=NULL) +
    theme(legend.justification="center", legend.position="bottom", axis.title.x=element_text(color="black")) +
    guides(fill=guide_legend(nrow=2))
    mb_d1


    # Donor 2
    mb_d2 <- ggplot(data=Mb2, aes(x=Phase, y=total_value, fill=variable)) +
    geom_bar(stat="identity", position = position_stack(vjust = 0.2)) +
    geom_text(aes(label=ifelse(total_value==0,"", sprintf("%.2f",total_value))), position=position_stack(vjust=0.5),size=3,color="white") +
    ylab("Donor 2 Metabolites (mM)") +
    xlab("Phase") +
    scale_fill_manual(values=thirtycolors, name=NULL) +
    theme(legend.justification="center", legend.position="bottom", axis.title.x=element_text(color="black")) +
    guides(fill=guide_legend(nrow=2))
    mb_d2


    # Donor 3
    mb_d3 <- ggplot(data=Mb3, aes(x=Phase, y=total_value, fill=variable)) +
    geom_bar(stat="identity") +
    geom_text(aes(label=ifelse(total_value==0,"", sprintf("%.2f",total_value))), position=position_stack(vjust=0.5),size=3,color="white") +
    ylab("Donor 3 Metabolites (mM)") +
    xlab("Phase") +
    scale_fill_manual(values=thirtycolors, name=NULL) +
    theme(legend.justification="center", legend.position="bottom", axis.title.x=element_text(color="black")) +
    guides(fill=guide_legend(nrow=2))
    mb_d3


    dpi <- 300
    tiff("Figure2.tiff",width=5*dpi,height=10*dpi,res=dpi,compression="lzw")
    cowplot::plot_grid(plot_grid(mb_d1,mb_d2,mb_d3,nrow=1,ncol=3,rel_heights=c(0.5,0.5,0.5,0.5),rel_widths=c(0.5,0.5,0.5,0.5),labels='AUTO',label_size=10,label_fontfamily='sans'),mylegendSCFA,nrow=2,rel_heights=c(1,1))
```

In [ ]:
```{r db-RDA for metabolites}

    Lumen_Metabolites_noNA <- na.omit(Lumen[,15:24])
    G4_metatable_cleaned <- G4_metatable[complete.cases(G4_metatable$ammonium_mM), ]

    Metabolite_dbrdail <- vegan::capscale(Lumen_Metabolites_noNA ~ Phase, G4_metatable_cleaned, dist="jaccard")
    summary(Metabolite_dbrdail)
    plot(Metabolite_dbrdail)

    Dim1 <- round(summary(Metabolite_dbrdail)$concont$importance[2,1]*100,0)
    Dim2 <- round(summary(Metabolite_dbrdail)$concont$importance[2,2]*100,0)

    Runadj <- RsquareAdj(Metabolite_dbrdail)$r.squared
    Radj <- RsquareAdj(Metabolite_dbrdail)$adj.r.squared


    plot(Metabolite_dbrdail,scaling=2,main="Triplot RDA datanormalised_subset ~ functional -scaling 2 ",type = "text")

    firstaxispointsvar <- summary(Metabolite_dbrdail)$centroids[,1]
    secondaxispointsvar <- summary(Metabolite_dbrdail)$centroids[,2]
    varpoints <- data.frame(cbind(firstaxispointsvar,secondaxispointsvar))
    varpointsarrow <- varpoints
    for(i in 1:nrow(varpointsarrow)){
      if (varpointsarrow$firstaxispointsvar[i]<0){varpointsarrow$firstaxispointsvar[i]=varpointsarrow$firstaxispointsvar[i]+0.07}
    else{varpointsarrow$firstaxispointsvar[i]=varpointsarrow$firstaxispointsvar[i]-0.07}
      if (varpointsarrow$secondaxispointsvar[i]<0){varpointsarrow$secondaxispointsvar[i]=varpointsarrow$secondaxispointsvar[i]+0.07}
    else{varpointsarrow$secondaxispointsvar[i]=varpointsarrow$secondaxispointsvar[i]-0.07}
    }
    rownames(varpoints) <- c("stabilisation","prebiotic","AB","ABwo")

    rdascoremet <- data.frame(vegan::scores(Metabolite_dbrdail,choices=1:2,scaling=2,display="species"))

    rdascores <- vegan::scores(Metabolite_dbrdail,choices=1:2,scaling=2,display="sites")
    firstaxispoints <- rdascores[,1]
    secondaxispoints <- rdascores[,2]
    points <- cbind(firstaxispoints,secondaxispoints,G4_metatable_cleaned)

    Met_plotobj <- ggplot(as.data.frame(points))+geom_point(aes(x=firstaxispoints,y=secondaxispoints, color=points$Donor),size=5) + scale_colour_manual(values=point_colors) + xlab(paste('CAP1',Dim1,"%"," ")) + ylab(paste('CAP2',Dim2,"%"," ")) +geom_text(data=varpoints,aes(x=firstaxispointsvar,y=secondaxispointsvar),size=5,color="blue",label=rownames(varpoints))+geom_segment(data=varpointsarrow,aes(x=0,xend=firstaxispointsvar,y=0,yend=secondaxispointsvar),color="blue",arrow=arrow(length=unit(0.03,"npc"))) + geom_text(data=rdascoremet,aes(x=CAP1,y=CAP2,label=rownames(rdascoremet)),size=5,color="red") + theme(legend.text=element_text(size=13))
    Met_plotobj
    ggsave(Met_plotobj, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Met_plotobj_donor", device = "png", width = 12, height = 9, limitsize=FALSE)
```

In [ ]:
```{r}
# compare metabolite concentrations between phase
acetate_stabilisation <- subset(Acetate, Phase=="stabilisation")
acetate_prebiotic <- subset(Acetate, Phase=="prebiotic")
acetate_AB <- subset(Acetate, Phase=="AB")
acetate_ABwo <- subset(Acetate, Phase=="ABwo")

wilcox_result_acetate1 <- wilcox.test(acetate_stabilisation$acetate,acetate_prebiotic$acetate)
wilcox_result_acetate2 <- wilcox.test(acetate_prebiotic$acetate,acetate_AB$acetate)
wilcox_result_acetate3 <- wilcox.test(acetate_AB$acetate,acetate_ABwo$acetate)

wilcox_result_acetate1
wilcox_result_acetate2
wilcox_result_acetate3

propionate_stabilisation <- subset(Propionate, Phase=="stabilisation")
propiopnate_prebiotic <- subset(Propionate, Phase=="prebiotic")
propiopnate_AB <- subset(Propionate, Phase=="AB")
propiopnate_ABwo <- subset(Propionate, Phase=="ABwo")

wilcox_result_propionate1 <- wilcox.test(propionate_stabilisation$propionate,propiopnate_prebiotic$propionate)
wilcox_result_propionate2 <- wilcox.test(propiopnate_prebiotic$propionate,propiopnate_AB$propionate)
wilcox_result_propionate3 <- wilcox.test(propiopnate_AB$propionate,propiopnate_ABwo$propionate)

wilcox_result_propionate1
wilcox_result_propionate2
wilcox_result_propionate3

isobutyrate_stabilisation <- subset(Isobutyrate, Phase=="stabilisation")
isobutyrate_prebiotic <- subset(Isobutyrate, Phase=="prebiotic")
isobutyrate_AB <- subset(Isobutyrate, Phase=="AB")

wilcox_result_isobutyrate1 <- wilcox.test(isobutyrate_stabilisation$isobutyrate,isobutyrate_prebiotic$isobutyrate)
wilcox_result_isobutyrate2 <- wilcox.test(isobutyrate_prebiotic$isobutyrate,isobutyrate_AB$isobutyrate)

wilcox_result_isobutyrate1
wilcox_result_isobutyrate2

butyrate_stabilisation <- subset(Butyrate, Phase=="stabilisation")
butyrate_prebiotic <- subset(Butyrate, Phase=="prebiotic")
butyrate_AB <- subset(Butyrate, Phase=="AB")

wilcox_result_butyrate1 <- wilcox.test(butyrate_stabilisation$butyrate,butyrate_prebiotic$butyrate)
wilcox_result_butyrate2 <- wilcox.test(butyrate_prebiotic$butyrate,butyrate_AB$butyrate)

wilcox_result_butyrate1
wilcox_result_butyrate2

isovalerate_stabilisation <- subset(Isovalerate, Phase=="stabilisation")
isovalerate_prebiotic <- subset(Isovalerate, Phase=="prebiotic")
isovalerate_AB <- subset(Isovalerate, Phase=="AB")

wilcox_result_isovalerate1 <- wilcox.test(isovalerate_stabilisation$isovalerate,isovalerate_prebiotic$isovalerate)
wilcox_result_isovalerate2 <- wilcox.test(isovalerate_prebiotic$isovalerate,isovalerate_AB$isovalerate)

wilcox_result_isovalerate1
wilcox_result_isovalerate2

valerate_stabilisation <- subset(Valerate, Phase=="stabilisation")
valerate_prebiotic <- subset(Valerate, Phase=="prebiotic")

wilcox_result_valerate1 <- wilcox.test(valerate_stabilisation$valerate,valerate_prebiotic$valerate)

wilcox_result_valerate1

ammonium_stabilisation <- subset(Ammonium, Phase=="stabilisation")
ammonium_prebiotic <- subset(Ammonium, Phase=="prebiotic")
ammonium_AB <- subset(Ammonium, Phase=="AB")
ammonium_ABwo <- subset(Ammonium, Phase=="ABwo")

wilcox_result_ammonium1 <- wilcox.test(ammonium_stabilisation$ammonium_mM,ammonium_prebiotic$ammonium_mM)
wilcox_result_ammonium2 <- wilcox.test(ammonium_prebiotic$ammonium_mM,ammonium_AB$ammonium_mM)
wilcox_result_ammonium3 <- wilcox.test(ammonium_AB$ammonium_mM,ammonium_ABwo$ammonium_mM)

wilcox_result_ammonium1
wilcox_result_ammonium2
wilcox_result_ammonium3
```

{B.2}

In [ ]:
```{r B.2: metabolites graph}

Acetate <- G4_metatable[,c(1, 6, 7, 15)]
Propionate <- G4_metatable[,c(1, 6, 7, 16)]
Isobutyrate <- G4_metatable[,c(1, 6, 7, 17)]
Butyrate <- G4_metatable[,c(1, 6, 7, 18)]
Isovalerate <- G4_metatable[,c(1, 6, 7, 19)]
Valerate <- G4_metatable[,c(1, 6, 7, 20)]
Isocaproate <- G4_metatable[,c(1, 6, 7, 21)]
Caproate <- G4_metatable[,c(1, 6, 7, 22)]
Heptanoate <- G4_metatable[,c(1, 6, 7, 23)]
Ammonium <- G4_metatable[,c(1, 6, 7, 24)]

# draw timepoint vs. qPCR_av for each donor
  Acetate_nona <- Acetate %>% filter(!is.na(acetate))
  Acetate_donor1 <- subset(Acetate_nona, Donor=='Donor 1')
  Acetate_donor2 <- subset(Acetate_nona, Donor=='Donor 2')
  Acetate_donor3 <- subset(Acetate_nona, Donor=='Donor 3')

  p111 <- ggplot(data=Acetate_donor1,aes(x=Timepoint, y=acetate)) + geom_line(color = "blue")
  p222 <- ggplot(data=Acetate_donor2,aes(x=Timepoint, y=acetate)) + geom_line(color = "blue")
  p333 <- ggplot(data=Acetate_donor3,aes(x=Timepoint, y=acetate)) + geom_line(color = "blue")
  grid.arrange(p111, p222, p333, ncol=1)


       #linegraphs
       plot <- ggplot(Acetate_nona,aes(x=Timepoint,y=acetate)) + geom_point(aes(colour=as.factor(Donor)))
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("acetate") + geom_vline(xintercept = c(8, 23, 30), linetype = "dashed", color = "gray") +ggtitle("Acetate") + theme(plot.title = element_text(size = 16)) + theme(axis.title.x = element_text(size = 16)) + theme(axis.title.y = element_text(size = 16))

       plot

  ggsave(plot, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Acetate",
       device = "png", width = 9, height = 5.5)

# draw timepoint vs. qPCR_av for each donor
  Propionate_nona <- Propionate %>% filter(!is.na(propionate))
  Propionate_donor1 <- subset(Propionate_nona, Donor=='Donor 1')
  Propionate_donor2 <- subset(Propionate_nona, Donor=='Donor 2')
  Propionate_donor3 <- subset(Propionate_nona, Donor=='Donor 3')

  p111 <- ggplot(data=Propionate_donor1,aes(x=Timepoint, y=propionate)) + geom_line(color = "blue")
  p222 <- ggplot(data=Propionate_donor2,aes(x=Timepoint, y=propionate)) + geom_line(color = "blue")
  p333 <- ggplot(data=Propionate_donor3,aes(x=Timepoint, y=propionate)) + geom_line(color = "blue")
  grid.arrange(p111, p222, p333, ncol=1)


       #linegraphs
       plot <- ggplot(Propionate_nona,aes(x=Timepoint,y=propionate)) + geom_point(aes(colour=as.factor(Donor)))
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("propionate") + geom_vline(xintercept = c(8, 23, 30), linetype = "dashed", color = "gray") +ggtitle("Propionate") + theme(plot.title = element_text(size = 16)) + theme(axis.title.x = element_text(size = 16)) + theme(axis.title.y = element_text(size = 16))

       plot

  ggsave(plot, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Propionate",
       device = "png", width = 9, height = 5.5)

# draw timepoint vs. qPCR_av for each donor
  Isobutyrate_nona <- Isobutyrate %>% filter(!is.na(isobutyrate))
  Isobutyrate_donor1 <- subset(Isobutyrate_nona, Donor=='Donor 1')
  Isobutyrate_donor2 <- subset(Isobutyrate_nona, Donor=='Donor 2')
  Isobutyrate_donor3 <- subset(Isobutyrate_nona, Donor=='Donor 3')

  p111 <- ggplot(data=Isobutyrate_donor1,aes(x=Timepoint, y=isobutyrate)) + geom_line(color = "blue")
  p222 <- ggplot(data=Isobutyrate_donor2,aes(x=Timepoint, y=isobutyrate)) + geom_line(color = "blue")
  p333 <- ggplot(data=Isobutyrate_donor3,aes(x=Timepoint, y=isobutyrate)) + geom_line(color = "blue")
  grid.arrange(p111, p222, p333, ncol=1)


       #linegraphs
       plot <- ggplot(Isobutyrate_nona,aes(x=Timepoint,y=isobutyrate)) + geom_point(aes(colour=as.factor(Donor)))
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("isobutyrate") + geom_vline(xintercept = c(8, 23, 30), linetype = "dashed", color = "gray") +ggtitle("Isobutyrate") + theme(plot.title = element_text(size = 16)) + theme(axis.title.x = element_text(size = 16)) + theme(axis.title.y = element_text(size = 16))

       plot

  ggsave(plot, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Isobutyrate",
       device = "png", width = 9, height = 5.5)

# draw timepoint vs. qPCR_av for each donor
  Butyrate_nona <- Butyrate %>% filter(!is.na(butyrate))
  Butyrate_donor1 <- subset(Butyrate_nona, Donor=='Donor 1')
  Butyrate_donor2 <- subset(Butyrate_nona, Donor=='Donor 2')
  Butyrate_donor3 <- subset(Butyrate_nona, Donor=='Donor 3')

  p111 <- ggplot(data=Butyrate_donor1,aes(x=Timepoint, y=butyrate)) + geom_line(color = "blue")
  p222 <- ggplot(data=Butyrate_donor2,aes(x=Timepoint, y=butyrate)) + geom_line(color = "blue")
  p333 <- ggplot(data=Butyrate_donor3,aes(x=Timepoint, y=butyrate)) + geom_line(color = "blue")
  grid.arrange(p111, p222, p333, ncol=1)


       #linegraphs
       plot <- ggplot(Butyrate_nona,aes(x=Timepoint,y=butyrate)) + geom_point(aes(colour=as.factor(Donor)))
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("butyrate") + geom_vline(xintercept = c(8, 23, 30), linetype = "dashed", color = "gray") +ggtitle("Butyrate") + theme(plot.title = element_text(size = 16)) + theme(axis.title.x = element_text(size = 16)) + theme(axis.title.y = element_text(size = 16))

       plot

  ggsave(plot, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Butyrate",
       device = "png", width = 9, height = 5.5)

# draw timepoint vs. qPCR_av for each donor
  Isovalerate_nona <- Isovalerate %>% filter(!is.na(isovalerate))
  Isovalerate_donor1 <- subset(Isovalerate_nona, Donor=='Donor 1')
  Isovalerate_donor2 <- subset(Isovalerate_nona, Donor=='Donor 2')
  Isovalerate_donor3 <- subset(Isovalerate_nona, Donor=='Donor 3')

  p111 <- ggplot(data=Isovalerate_donor1,aes(x=Timepoint, y=isovalerate)) + geom_line(color = "blue")
  p222 <- ggplot(data=Isovalerate_donor2,aes(x=Timepoint, y=isovalerate)) + geom_line(color = "blue")
  p333 <- ggplot(data=Isovalerate_donor3,aes(x=Timepoint, y=isovalerate)) + geom_line(color = "blue")
  grid.arrange(p111, p222, p333, ncol=1)


       #linegraphs
       plot <- ggplot(Isovalerate_nona,aes(x=Timepoint,y=isovalerate)) + geom_point(aes(colour=as.factor(Donor)))
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("isovalerate") + geom_vline(xintercept = c(8, 23, 30), linetype = "dashed", color = "gray") +ggtitle("Isovalerate") + theme(plot.title = element_text(size = 16)) + theme(axis.title.x = element_text(size = 16)) + theme(axis.title.y = element_text(size = 16))
       plot

  ggsave(plot, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Isovalerate",
       device = "png", width = 9, height = 5.5)


# draw timepoint vs. qPCR_av for each donor
  Valerate_nona <- Valerate %>% filter(!is.na(valerate))
  Valerate_donor1 <- subset(Valerate_nona, Donor=='Donor 1')
  Valerate_donor2 <- subset(Valerate_nona, Donor=='Donor 2')
  Valerate_donor3 <- subset(Valerate_nona, Donor=='Donor 3')

  p111 <- ggplot(data=Valerate_donor1,aes(x=Timepoint, y=valerate)) + geom_line(color = "blue")
  p222 <- ggplot(data=Valerate_donor2,aes(x=Timepoint, y=valerate)) + geom_line(color = "blue")
  p333 <- ggplot(data=Valerate_donor3,aes(x=Timepoint, y=valerate)) + geom_line(color = "blue")
  grid.arrange(p111, p222, p333, ncol=1)


       #linegraphs
       plot <- ggplot(Valerate_nona,aes(x=Timepoint,y=valerate)) + geom_point(aes(colour=as.factor(Donor)))
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("valerate") + geom_vline(xintercept = c(8, 23, 30), linetype = "dashed", color = "gray") +ggtitle("Valerate") + theme(plot.title = element_text(size = 16)) + theme(axis.title.x = element_text(size = 16)) + theme(axis.title.y = element_text(size = 16))

       plot

  ggsave(plot, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Valerate",
       device = "png", width = 9, height = 5.5)

# draw timepoint vs. qPCR_av for each donor
  Ammonium_nona <- Ammonium %>% filter(!is.na(ammonium_mM))
  Ammonium_donor1 <- subset(Ammonium_nona, Donor=='Donor 1')
  Ammonium_donor2 <- subset(Ammonium_nona, Donor=='Donor 2')
  Ammonium_donor3 <- subset(Ammonium_nona, Donor=='Donor 3')

  p111 <- ggplot(data=Ammonium_donor1,aes(x=Timepoint, y=ammonium_mM)) + geom_line(color = "blue")
  p222 <- ggplot(data=Ammonium_donor2,aes(x=Timepoint, y=ammonium_mM)) + geom_line(color = "blue")
  p333 <- ggplot(data=Ammonium_donor3,aes(x=Timepoint, y=ammonium_mM)) + geom_line(color = "blue")
  grid.arrange(p111, p222, p333, ncol=1)


       #linegraphs
       plot <- ggplot(Ammonium_nona,aes(x=Timepoint,y=ammonium_mM)) + geom_point(aes(colour=as.factor(Donor)))
       plot <- plot + geom_line(aes(colour=as.factor(Donor)),size=1.5,alpha=0.4,show.legend = FALSE)
       plot <- papertheme(plot)
       plot <- plot + xlab("Time") +ylab("ammonium_mM") + geom_vline(xintercept = c(8, 23, 30), linetype = "dashed", color = "gray") +ggtitle("Ammonium") + theme(plot.title = element_text(size = 16)) + theme(axis.title.x = element_text(size = 16)) + theme(axis.title.y = element_text(size = 16))

       plot

  ggsave(plot, path = "/data/gent/458/vsc45876/2023_HAM/GroupWork/Assignment2_Output", filename = "Ammonium",
       device = "png", width = 9, height = 5.5)

```

In [ ]:
```{r}
ABwo1 <- subset(Propionate,Timepoint >= 31 & Timepoint <= 38, select = c("propionate"))
ABwo1_avg <- mean(ABwo1$propionate, na.rm=T)

ABwo2 <- subset(Propionate,Timepoint >= 39, select = c("propionate"))
ABwo2_avg <- mean(ABwo2$propionate, na.rm=T)

ABwo2_avg - ABwo1_avg
```

{C.1}

In [ ]:
```{r C.1}
    #Diversity indices: plotting per sample
    ## Extrapolated richness
    div <- estimateD(t(G4_ASVtable), datatype="abundance", base="coverage", level=0.985, conf=0.95)
    div_H0 <- div %>% subset(Order.q=="0")
    div_H1 <- div %>% subset(Order.q=="1")

    ##add metadata to plot diversity per sample
    G4_metatable$Samplenumber_corr <- G4_metatable$Samplenumber +1
    div_H0_full <- cbind(noNA, div_H0)
    div_H1_full <- cbind(noNA, div_H1)

    # Hill 0 Richness
    ## Linegraphs
    plot_richness <- ggplot(div_H0_full,aes(x=Timepoint,y=qD)) + geom_pointrange(aes(ymin=qD.LCL,ymax=qD.UCL,colour=Lumen_Bran), position=position_jitter(width=0,height=0.2),size=0.2)
    plot_richness <- plot_richness + geom_line(aes(colour=Lumen_Bran), size=1.5, alpha=0.4, show.legend=FALSE) + facet_grid(~Donor)
    plot_richness <- papertheme(plot_richness)
    plot_richness <- plot_richness + xlab("Time") + ylab("Richness at extrapolated sample coverage") + geom_vline(xintercept=c(8, 23, 30), linetype="dashed", color="gray")
    plot_richness <- plot_richness + theme(axis.title.x = element_text(size = 15)) + theme(axis.title.y = element_text(size = 15))
    plot_richness
    # ggsave(plot_richness, path = output_path, filename = "Richness_C1",
    #    device = "png", width = 9, height = 5.5)

    # Hill 1 Shannon diversity
    ## Linegraphs
    plot_shannon <- ggplot(div_H1_full,aes(x=Timepoint,y=qD)) + geom_pointrange(aes(ymin=qD.LCL,ymax=qD.UCL,colour=Lumen_Bran), position=position_jitter(width=0,height=0.2),size=0.2)
    plot_shannon <- plot_shannon + geom_line(aes(colour=Lumen_Bran), size=1.5, alpha=0.4, show.legend=FALSE) + facet_grid(~Donor)
    plot_shannon <- papertheme(plot_shannon)
    plot_shannon <- plot_shannon + xlab("Time") + ylab("Shannon diversity at extrapolated sample coverage") + geom_vline(xintercept=c(8, 23, 30), linetype="dashed", color="gray")
    plot_shannon <- plot_shannon + theme(axis.title.x = element_text(size = 15)) + theme(axis.title.y = element_text(size = 15))
    plot_shannon
    # ggsave(plot_shannon, path = output_path, filename = "Shannon_C1",
    #    device = "png", width = 9, height = 5.5)
```

{C.2}

In [ ]:
 ```{r C.2}
  # Rarefaction curve
  set.seed=711
  datarare <- t(G4_ASVtable)
  # Find minimal total number of reads of all samples = mimimal number of species
  raremax <- min(colSums(datarare))
  # Draw rarefaction curves
  rarecurve(round(t(datarare)), step = 20, sample = raremax, col = thirtycolors, cex = 0.6)
```

{C.3} Calculate the proportional microbial community abundance profiles at genus level and plot the evolution of the luminal and bran-attached community over time for th emost abundant taxa.


In [ ]:
```{r C.3}
    # Aggregate at Genus level
    G4_genusASVtable <- aggregate(. ~ Genus, data=G4_ASVtax[,-c(1:5)], FUN=sum)
    rownames(G4_genusASVtable) <- G4_genusASVtable[,1]
    G4_genusASVtable <- G4_genusASVtable[,-1]
    G4_genusASVtable_prop <- sweep(G4_genusASVtable, 2, colSums(G4_genusASVtable),'/')
    G4_genus_prop_rowsum <- as.matrix(rowSums(G4_genusASVtable_prop))

    # Find the most abundant genus
    G4_genus_prop_rowsum[which.max(G4_genus_prop_rowsum),]

    # Extract only the most abundance taxa data (Prevotella_9)
    most_abundant_taxa <- subset(G4_genustable_bar, variable == "Prevotella_9")

    plot_mostabundance <- ggplot(most_abundant_taxa,aes(x=Timepoint,y=value))
    plot_mostabundance <- plot_mostabundance + geom_line(aes(colour=Lumen_Bran), size=1.5, alpha=0.4, show.legend=TRUE) + facet_grid(~Donor)
    plot_mostabundance <- papertheme(plot_mostabundance)
    plot_mostabundance <- plot_mostabundance + xlab("Time") + ylab("Prevotella_9 Abundance") + geom_vline(xintercept=c(8, 23, 30), linetype = "dashed", color="gray")

    plot_mostabundance
```

{C.4}

In [ ]:
```{r db-RDA for Lumen}
noNA_rda <- noNA
noNA_rda$Donor_Phase <- paste(noNA_rda$Donor, noNA_rda$Phase, sep="_")
noNA_rda$Donor_Phase <- factor(noNA_rda$Donor_Phase, levels=c("Donor 1_stabilisation","Donor 1_prebiotic","Donor 1_AB","Donor 1_ABwo","Donor 2_stabilisation","Donor 2_prebiotic","Donor 2_AB","Donor 2_ABwo","Donor 3_stabilisation","Donor 3_prebiotic","Donor 3_AB","Donor 3_ABwo"))
Lumen_noNA <- subset(noNA_rda, Lumen_Bran=='Lumen')
shifted_noNA <- rbind(noNA_rda[-(1:3), ], noNA_rda[1:3, ])
Lumen_prop <- G4_genusASVtable_prop[,shifted_noNA$Lumen_Bran == 'Lumen']

Lumen_dbrdail <- vegan::capscale(t(Lumen_prop) ~ Donor_Phase, Lumen_noNA, dist="bray")
summary(Lumen_dbrdail)
plot(Lumen_dbrdail)

Dim1 <- round(summary(Lumen_dbrdail)$concont$importance[2,1]*100,0)
Dim2 <- round(summary(Lumen_dbrdail)$concont$importance[2,2]*100,0)

Runadj <- RsquareAdj(Lumen_dbrdail)$r.squared
Radj <- RsquareAdj(Lumen_dbrdail)$adj.r.squared
anova(Lumen_dbrdail, by="terms", permutations = 999)

Lumen_abundant_Genus <- as.data.frame(rownames_to_column(data.frame((Lumen_prop))) %>% top_n(29,rowSums((Lumen_prop))))$rowname

plot(Lumen_dbrdail,scaling=2,main="Triplot RDA datanormalised_subset ~ functional -scaling 2 ",type = "text")

firstaxispointsvar <- summary(Lumen_dbrdail)$centroids[,1]
secondaxispointsvar <- summary(Lumen_dbrdail)$centroids[,2]
varpoints <- data.frame(cbind(firstaxispointsvar,secondaxispointsvar))
varpointsarrow <- varpoints
    for(i in 1:nrow(varpointsarrow)){
      if (varpointsarrow$firstaxispointsvar[i]<0){varpointsarrow$firstaxispointsvar[i]=varpointsarrow$firstaxispointsvar[i]+0.07}
    else{varpointsarrow$firstaxispointsvar[i]=varpointsarrow$firstaxispointsvar[i]-0.07}
      if (varpointsarrow$secondaxispointsvar[i]<0){varpointsarrow$secondaxispointsvar[i]=varpointsarrow$secondaxispointsvar[i]+0.07}
    else{varpointsarrow$secondaxispointsvar[i]=varpointsarrow$secondaxispointsvar[i]-0.07}
    }
    rownames(varpoints) <- c("Donor 1_stabilisation","Donor 1_prebiotic","Donor 1_AB","Donor 1_ABwo","Donor 2_stabilisation","Donor 2_prebiotic","Donor 2_AB","Donor 2_ABwo","Donor 3_stabilisation","Donor 3_prebiotic","Donor 3_AB","Donor 3_ABwo")

    rdascorespecies <- data.frame(vegan::scores(Lumen_dbrdail,choices=1:2,scaling=2,display="species"))
    rdascorespecies <- rdascorespecies[rownames(rdascorespecies)%in%Lumen_abundant_Genus,]

    rdascores <- vegan::scores(Lumen_dbrdail,choices=1:2,scaling=2,display="sites")
    firstaxispoints <- rdascores[,1]
    secondaxispoints <- rdascores[,2]
    points <- cbind(firstaxispoints,secondaxispoints,Lumen_noNA)

    Lumen_plotobj <- ggplot(as.data.frame(points))+geom_point(aes(x=firstaxispoints,y=secondaxispoints, color=points$Donor_Phase),size=5) + scale_colour_manual(values=thirtycolors) + xlab(paste('CAP1',Dim1,"%"," ")) + ylab(paste('CAP2',Dim2,"%"," ")) +geom_text(data=varpoints,aes(x=firstaxispointsvar,y=secondaxispointsvar),size=5,color="blue",label=rownames(varpoints))+geom_segment(data=varpointsarrow,aes(x=0,xend=firstaxispointsvar,y=0,yend=secondaxispointsvar),color="blue",arrow=arrow(length=unit(0.03,"npc"))) + geom_text(data=rdascorespecies,aes(x=CAP1,y=CAP2,label=rownames(rdascorespecies)),size=5,color="red") + theme(legend.text=element_text(size=13))
    Lumen_plotobj
  ggsave(Lumen_plotobj, path = output_path, filename = "Lumen_plotobj", device = "png", width = 12, height = 9, limitsize=FALSE)
```


```{r db-RDA for Bran}
Bran_noNA <- subset(noNA_rda, Lumen_Bran=='Bran')
Bran_prop <- G4_genusASVtable_prop[,shifted_noNA$Lumen_Bran=='Bran']

Bran_dbrdail <- vegan::capscale(t(Bran_prop) ~ Donor_Phase, Bran_noNA, dist="bray")
summary(Bran_dbrdail)
plot(Bran_dbrdail)

Dim1 <- round(summary(Bran_dbrdail)$concont$importance[2,1]*100,0)
Dim2 <- round(summary(Bran_dbrdail)$concont$importance[2,2]*100,0)

Runadj <- RsquareAdj(Bran_dbrdail)$r.squared
Radj <- RsquareAdj(Bran_dbrdail)$adj.r.squared
anova(Bran_dbrdail, by="terms", permutations = 999)

Bran_abundant_Genus <- as.data.frame(rownames_to_column(data.frame((Bran_prop))) %>% top_n(29,rowSums((Bran_prop))))$rowname

plot(Bran_dbrdail,scaling=2,main="Triplot RDA datanormalised_subset ~ functional -scaling 2 ",type = "text")

firstaxispointsvar <- summary(Bran_dbrdail)$centroids[,1]
secondaxispointsvar <- summary(Bran_dbrdail)$centroids[,2]
varpoints <- data.frame(cbind(firstaxispointsvar,secondaxispointsvar))
varpointsarrow <- varpoints
    for(i in 1:nrow(varpointsarrow)){
      if (varpointsarrow$firstaxispointsvar[i]<0){varpointsarrow$firstaxispointsvar[i]=varpointsarrow$firstaxispointsvar[i]+0.07}
    else{varpointsarrow$firstaxispointsvar[i]=varpointsarrow$firstaxispointsvar[i]-0.07}
      if (varpointsarrow$secondaxispointsvar[i]<0){varpointsarrow$secondaxispointsvar[i]=varpointsarrow$secondaxispointsvar[i]+0.07}
    else{varpointsarrow$secondaxispointsvar[i]=varpointsarrow$secondaxispointsvar[i]-0.07}
    }
    rownames(varpoints) <- c("Donor 1_prebiotic","Donor 1_AB","Donor 1_ABwo","Donor 2_prebiotic","Donor 2_AB","Donor 2_ABwo","Donor 3_prebiotic","Donor 3_AB","Donor 3_ABwo")

    rdascorespecies <- data.frame(vegan::scores(Bran_dbrdail,choices=1:2,scaling=2,display="species"))
    rdascorespecies <- rdascorespecies[rownames(rdascorespecies)%in%Bran_abundant_Genus,]

    rdascores <- vegan::scores(Bran_dbrdail,choices=1:2,scaling=2,display="sites")
    firstaxispoints <- rdascores[,1]
    secondaxispoints <- rdascores[,2]
    points <- cbind(firstaxispoints,secondaxispoints,Bran_noNA)

    Bran_plotobj <- ggplot(as.data.frame(points))+geom_point(aes(x=firstaxispoints,y=secondaxispoints, color=points$Donor_Phase),size=5) + scale_colour_manual(values=thirtycolors) + xlab(paste('CAP1',Dim1,"%"," ")) + ylab(paste('CAP2',Dim2,"%"," ")) +geom_text(data=varpoints,aes(x=firstaxispointsvar,y=secondaxispointsvar),size=5,color="blue",label=rownames(varpoints))+geom_segment(data=varpointsarrow,aes(x=0,xend=firstaxispointsvar,y=0,yend=secondaxispointsvar),color="blue",arrow=arrow(length=unit(0.03,"npc"))) + geom_text(data=rdascorespecies,aes(x=CAP1,y=CAP2,label=rownames(rdascorespecies)),size=5,color="red") + theme(legend.text=element_text(size=13))
    Bran_plotobj
  ggsave(Bran_plotobj, path = output_path, filename = "Bran_plotobj", device = "png", width = 12, height = 9, limitsize=FALSE)
```


{C.5}


In [ ]:
```{r C.5.step 10: DESeq,cache=TRUE,warning=FALSE,message=FALSE,dpi=dpi,out.width="100%"}
# Create DESeq dataset
##combining ASV + genera
G4_test <- cbind(G4_ASVtable,t(G4_genusASVtable))
G4_test <- merge(G4_metatable, as.data.frame(G4_test),by.x="row.names",by.y="row.names",all.x=FALSE,all.y=TRUE)
G4_testmetadata <- G4_test[,c(1:ncol(G4_metatable))]
G4_testmetadata <- G4_testmetadata[order(G4_testmetadata$Timepoint),]

G4_testmetadata_lumen <- subset(G4_testmetadata, Lumen_Bran == "Lumen")
G4_testmetadata_bran <- subset(G4_testmetadata, Lumen_Bran == "Bran")

G4_test_lumen <- subset(G4_test, Lumen_Bran == "Lumen")
G4_test_bran <- subset(G4_test, Lumen_Bran == "Bran")
G4_test_lumen <- G4_test_lumen[,-c(1:31)]
G4_test_bran <- G4_test_bran[,-c(1:31)]

## Lumen
G4_deseqdata_lumen <- as.matrix(t(G4_test_lumen)+1) #to remove 0
G4_deseqdata_lumen <- DESeqDataSetFromMatrix(round(G4_deseqdata_lumen),colData=G4_testmetadata_lumen,design= ~Phase)
G4_deseqdata_lumen <- estimateSizeFactors(G4_deseqdata_lumen)
G4_deseqdata_lumen <- estimateDispersions(G4_deseqdata_lumen,fitType="local")

# Overall significant effect of the different nutrient conditions
G4_test_lumendiff<- DESeq(G4_deseqdata_lumen,test="Wald", betaPrior=FALSE)
G4_restestdiff <- results(G4_test_lumendiff,cooksCutoff=FALSE, independentFiltering=FALSE)
resultsNames(G4_test_lumendiff)
G4_restestdiffordered <- G4_restestdiff[order(G4_restestdiff$padj),]
summary(G4_restestdiff)
levels(as.factor(G4_testmetadata_lumen$Phase))

# Pairwise significant differences
## Phase_prebiotic_vs_stabilisation
resPre_Stab_lumen <- results(G4_test_lumendiff,contrast=c("Phase","prebiotic","stabilisation"),test="Wald",alpha=0.05, cooksCutoff=FALSE, independentFiltering=FALSE)
resPre_Stab_lumen <- lfcShrink(G4_test_lumendiff,contrast=c("Phase","prebiotic","stabilisation"),res=resPre_Stab_lumen,type="ashr")
resPre_Stab_lumen_sign <- resPre_Stab_lumen$padj<0.05
resPre_Stab_lumen_signOTU <- na.omit(colnames(G4_test_lumen)[resPre_Stab_lumen_sign==TRUE])
resPre_Stab_lumenvol <- volcanokim(resPre_Stab_lumen,taxlevel="species")+ggtitle("stabilisation \u2013 prebiotic")
mylegend_volcano_prestab <- get_legend(resPre_Stab_lumenvol)
resPre_Stab_lumenvol <-resPre_Stab_lumenvol+theme(legend.position="none")
resPre_Stab_lumenvol

# Phase_AB_vs_prebiotic
resAB_Pre_lumen <- results(G4_test_lumendiff,contrast=c("Phase","AB","prebiotic"),test="Wald",alpha=0.05, cooksCutoff=FALSE, independentFiltering=FALSE)
resAB_Pre_lumen <- lfcShrink(G4_test_lumendiff,contrast=c("Phase","AB","prebiotic"),res=resAB_Pre_lumen,type="ashr")
resAB_Pre_lumen_sign <- resAB_Pre_lumen$padj<0.05
resAB_Pre_lumen_signOTU <- na.omit(colnames(G4_test_lumen)[resAB_Pre_lumen_sign==TRUE])
resAB_Pre_lumenvol <- volcanokim(resAB_Pre_lumen,taxlevel="species")+ggtitle("AB \u2013 prebiotic")
mylegend_volcano_ABpre <- get_legend(resAB_Pre_lumenvol)
resAB_Pre_lumenvol <- resAB_Pre_lumenvol+theme(legend.position="none")
resAB_Pre_lumenvol

## Phase_ABwo_vs_AB
rresABwo_AB_lumen <- results(G4_test_lumendiff,contrast=c("Phase","ABwo","AB"),test="Wald",alpha=0.05, cooksCutoff=FALSE, independentFiltering=FALSE)
rresABwo_AB_lumen <- lfcShrink(G4_test_lumendiff,contrast=c("Phase","ABwo","AB"),res=rresABwo_AB_lumen,type="ashr")
rresABwo_AB_lumen_sign <- rresABwo_AB_lumen$padj<0.05
rresABwo_AB_lumen_signOTU <- na.omit(colnames(G4_test_lumen)[rresABwo_AB_lumen_sign==TRUE])
rresABwo_AB_lumenvol <- volcanokim(rresABwo_AB_lumen,taxlevel="species")+ggtitle("ABwo \u2013 AB")
mylegend_volcano_ABwoAB <- get_legend(rresABwo_AB_lumenvol)
rresABwo_AB_lumenvol <-rresABwo_AB_lumenvol+theme(legend.position="none")
rresABwo_AB_lumenvol

# Concatenate all pairwise comparisons
##making dataframe with all sign ASVs + genera
G4_otunames_lumen <- unique(c(resAB_Pre_lumen_signOTU,rresABwo_AB_lumen_signOTU, resPre_Stab_lumen_signOTU))
G4_otunames_lumen <- na.omit(G4_otunames_lumen)
G4_phylumname <- mapvalues(G4_otunames_lumen,from=rownames(G4_Taxtable),to=as.character(G4_Taxtable$Phylum))
G4_phylumname <- mapvalues(G4_phylumname,from=G4_Taxtable$Genus,to=as.character(G4_Taxtable$Phylum))
G4_genusname <- mapvalues(G4_phylumname,from=rownames(G4_Taxtable),to=as.character(G4_Taxtable$Genus))
G4_combinedgenus_phylum_lumen <- data.frame(genus=G4_genusname,phylum=G4_phylumname,otuname=G4_otunames_lumen)

##adding p-values for all pairwise comparisons for each sign  ASV+genus + assigning letters
G4_saveotu_lumen <- data.frame()
for (i in G4_otunames_lumen){
pvalueAB_Pre <- resAB_Pre_lumen[i,]$padj
names(pvalueAB_Pre) <- c("Prebiotic-ABwo")
pvalueABwo_AB <- rresABwo_AB_lumen[i,]$padj
names(pvalueABwo_AB) <- c("AB-ABwo")
pvaluePre_Stab <- resPre_Stab_lumen[i,]$padj
names(pvaluePre_Stab) <- c("Prebiotic-Stabilization")
G4_pvaluetotal <- c(pvalueAB_Pre,pvalueABwo_AB,pvaluePre_Stab)
G4_letters <- multcompView::multcompLetters(G4_pvaluetotal)$"Letters"

#making dataframe with the counts of the sign ASVs/genera in all conditions + letters for annotating sign differences
G4_diffotu <- plotCounts(G4_test_lumendiff,gene=i,intgroup="Phase",returnData=TRUE)
G4_diffotu <- cbind(G4_diffotu,G4_testmetadata_lumen)
G4_diffotu <- mutate(G4_diffotu,phylum=rep(G4_combinedgenus_phylum_lumen[G4_combinedgenus_phylum_lumen$otuname==i,]$phylum,nrow(G4_diffotu)),genus=rep(G4_combinedgenus_phylum_lumen[G4_combinedgenus_phylum_lumen$otuname==i,]$genus,nrow(G4_diffotu)),otu=i)
G4_diffotu$labels<- mapvalues(G4_diffotu$Phase,from=unique(G4_diffotu$Phase),to=c(G4_letters[4],G4_letters[1],G4_letters[3],G4_letters[2]))
G4_saveotu_lumen <- rbind(G4_saveotu_lumen,G4_diffotu)
}

G4_saveotu_lumen <- ddply(G4_saveotu_lumen,.(Phase,otu),mutate, Q1=quantile(log(count),1/4), Q3=quantile(log(count),3/4),IQR=Q3-Q1,upper.limit=Q3+1.5*IQR,lower.limit=Q1-1.5*IQR)

# Order data in subpanels of multipanel graph
##Only working with genera
G4_otunames_lumen_reorder <- gtools::mixedsort(G4_otunames_lumen[grep("ASV",G4_otunames_lumen)])
G4_genus_names<-G4_otunames_lumen[!G4_otunames_lumen %in%  G4_otunames_lumen[grep("ASV",G4_otunames_lumen)]]

G4_saveotu_lumen1<-G4_saveotu_lumen[G4_saveotu_lumen$otu%in%G4_genus_names[1:14],]
G4_saveotu_lumen1<- G4_saveotu_lumen1[gtools::mixedorder(G4_saveotu_lumen1$otu),]
G4_saveotu_lumen1$otu <- DescTools::reorder.factor(x=factor(G4_saveotu_lumen1$otu))

G4_saveotu_lumen2<-G4_saveotu_lumen[G4_saveotu_lumen$otu%in%G4_genus_names[15:26],]
G4_saveotu_lumen2<- G4_saveotu_lumen2[gtools::mixedorder(G4_saveotu_lumen2$otu),]
G4_saveotu_lumen2$otu <- DescTools::reorder.factor(x=factor(G4_saveotu_lumen2$otu))


## Color strips by phylum level classification
G4_saveotu_lumenfillphylum <- rbind(G4_saveotu_lumen1,G4_saveotu_lumen2)
G4_saveotu_lumenfillphylum$fill <- mapvalues(G4_saveotu_lumenfillphylum$phylum,from=unique(G4_saveotu_lumenfillphylum$phylum),to=c("#953d6aB3","#cecb99","#5d6295B3","#837239B3","#557d7aB3"))
G4_saveotu_lumenfillphylum <-G4_saveotu_lumenfillphylum[!duplicated(G4_saveotu_lumenfillphylum$otu),]

#plotting to save the legend for future plots
plot_C10_lumen <- ggplot(G4_saveotu_lumenfillphylum,aes(x=Phase,y=count,fill=G4_saveotu_lumenfillphylum$phylum))+ geom_boxplot() + scale_fill_manual(breaks=unique(G4_saveotu_lumenfillphylum$phylum),values=c("#953d6aB3","#cecb99","#5d6295B3","#837239B3","#557d7aB3"),name="Phylum")+theme(legend.position="bottom",legend.text=element_text(face='italic'))+guides(fill=guide_legend(nrow=3))
striplegend <- get_legend(plot_C10_lumen)
G4_saveotu_lumenfillphylum1 <- G4_saveotu_lumenfillphylum[G4_saveotu_lumenfillphylum$otu%in%G4_saveotu_lumen1$otu,]
G4_saveotu_lumenfillphylum2 <- G4_saveotu_lumenfillphylum[G4_saveotu_lumenfillphylum$otu%in%G4_saveotu_lumen2$otu,]


## Create side-by-side box plots
plot_C10_lumen <- ggplot(G4_saveotu_lumen1,aes(x=Phase,y=count,fill=Phase))+ geom_boxplot()+geom_point(data=G4_saveotu_lumen1[log(G4_saveotu_lumen1$count)>G4_saveotu_lumen1$upper.limit|log(G4_saveotu_lumen1$count)<G4_saveotu_lumen1$lower.limit,],aes(x=Phase,y=count,color=Phase),show.legend=FALSE)
plot_C10_lumen <- plot_C10_lumen +scale_y_continuous(labels=scientific,trans='log10',breaks = trans_breaks('log10', function(x) 10^x))+ylab("Normalised log transformed counts") +facet_grid(.~otu,drop=TRUE,scales='free')
plot_C10_lumen <- papertheme(plot_C10_lumen)+theme(axis.title.x = element_blank()) + scale_fill_manual(values=point_colors,name=NULL)+ scale_color_manual(values=point_colors,name=NULL)+ theme(legend.justification = "center") +theme(strip.text.x=element_text(size=10, face = "bold"),axis.text.x = element_blank(),axis.title.y = element_text(color="white"))+geom_text(aes(label=labels,x=Phase,y=max(count)+10))
mylegenddeseqpre <- get_legend(plot_C10_lumen)
deseqsaveotu1_lumen <-plot_C10_lumen+ theme(legend.position = "none")
deseqsaveotu1strips_lumen<- ggplot_gtable(ggplot_build(deseqsaveotu1_lumen))
stripr <- which(grepl('strip-t',deseqsaveotu1strips_lumen$layout$name))
fills <- as.character(G4_saveotu_lumenfillphylum1$fill)
k <- 1
for (i in stripr) {
  j <- which(grepl('rect', deseqsaveotu1strips_lumen$grobs[[i]]$grobs[[1]]$childrenOrder))
  deseqsaveotu1strips_lumen$grobs[[i]]$grobs[[1]]$children[[j]]$gp$fill <- fills[k]
  k <- k+1
}

plot_C10_lumen <- ggplot(G4_saveotu_lumen2,aes(x=Phase,y=count,fill=Phase))+ geom_boxplot()+geom_point(data=G4_saveotu_lumen2[log(G4_saveotu_lumen2$count)>G4_saveotu_lumen2$upper.limit|log(G4_saveotu_lumen2$count)<G4_saveotu_lumen2$lower.limit,],aes(x=Phase,y=count,color=Phase),show.legend=FALSE)
plot_C10_lumen <- plot_C10_lumen +scale_y_continuous(labels=scientific,trans='log10',breaks = trans_breaks('log10', function(x) 10^x))+ylab("Normalised log transformed counts") +facet_grid(.~otu,drop=TRUE,scales='free')
plot_C10_lumen <- papertheme(plot_C10_lumen)+theme(axis.title.x = element_blank()) + scale_fill_manual(values=point_colors,name=NULL)+ scale_color_manual(values=point_colors,name=NULL)+ theme(legend.justification = "center") +theme(strip.text.x=element_text(size=10, face="bold"),axis.text.x = element_blank(),axis.title.y = element_text(color="white"))+geom_text(aes(label=labels,x=Phase,y=max(count)+10))
deseqGsaveotu2_lumen <-plot_C10_lumen+ theme(legend.position = "none")
deseqsaveotu2strips_lumen <- ggplot_gtable(ggplot_build(deseqGsaveotu2_lumen))
stripr <- which(grepl('strip-t',deseqsaveotu2strips_lumen$layout$name))
fills <- as.character(G4_saveotu_lumenfillphylum2$fill)
k <- 1
for (i in stripr) {
  j <- which(grepl('rect', deseqsaveotu2strips_lumen$grobs[[i]]$grobs[[1]]$childrenOrder))
  deseqsaveotu2strips_lumen$grobs[[i]]$grobs[[1]]$children[[j]]$gp$fill <- fills[k]
  k <- k+1
}


##combining all plots and writing out to tiff file in output path
tiff(paste0(output_path,"DESeq_Genus.tiff"), width=12*dpi, height=16*dpi, res=dpi, compression="lzw")
p_C10_lumen <- ggdraw(plot_grid(plot_grid(deseqsaveotu1strips_lumen,deseqsaveotu2strips_lumen,ncol=1,rel_heights = c(0.25,0.25,0.25,0.25)),plot_grid(mylegenddeseqpre,striplegend,ncol=2,rel_widths=c(0.5,0.5)),ncol=1,rel_heights=c(0.7,0.1)))+draw_label("Normalised log transformed counts",0.012,0.55,angle=90,size=18)
dev.off()
p_C10_lumen
#ggsave(p_C10_lumen, path = output_path, filename = "Lumen_box", device = "png", width = 18, height = 12, limitsize=FALSE)
```



```{r C.5.Bran}
# Bran

G4_deseqdata_bran <- as.matrix(t(G4_test_bran)+1) #to remove 0
G4_deseqdata_bran <- DESeqDataSetFromMatrix(round(G4_deseqdata_bran),colData=G4_testmetadata_bran,design= ~Phase)
G4_deseqdata_bran <- estimateSizeFactors(G4_deseqdata_bran)
G4_deseqdata_bran <- estimateDispersions(G4_deseqdata_bran,fitType="local")

# Overal significant effect of the different nutrient conditions
G4_test_brandiff<- DESeq(G4_deseqdata_bran,test="Wald", betaPrior=FALSE)
G4_restestdiff <- results(G4_test_brandiff,cooksCutoff=FALSE, independentFiltering=FALSE)
resultsNames(G4_test_brandiff)
G4_restestdiffordered <- G4_restestdiff[order(G4_restestdiff$padj),]
summary(G4_restestdiff)
levels(as.factor(G4_testmetadata_bran$Phase))

# Pairwise significant differences

## Phase_AB_vs_prebiotic
resAB_Pre_bran <- results(G4_test_brandiff,contrast=c("Phase","AB","prebiotic"),test="Wald",alpha=0.05, cooksCutoff=FALSE, independentFiltering=FALSE)
resAB_Pre_bran <- lfcShrink(G4_test_brandiff,contrast=c("Phase","AB","prebiotic"),res=resAB_Pre_bran,type="ashr")
resAB_Pre_bran_sign <- resAB_Pre_bran$padj<0.05
resAB_Pre_bran_signOTU <- na.omit(colnames(G4_test_bran)[resAB_Pre_bran_sign==TRUE])
resAB_Pre_branvol <- volcanokim(resAB_Pre_bran,taxlevel="species")+ggtitle("AB \u2013 prebiotic")
mylegend_volcano_ABpre <- get_legend(resAB_Pre_branvol)
resAB_Pre_branvol <- resAB_Pre_branvol+theme(legend.position="none")

## Phase_ABwo_vs_AB
rresABwo_AB_bran <- results(G4_test_brandiff,contrast=c("Phase","ABwo","AB"),test="Wald",alpha=0.05, cooksCutoff=FALSE, independentFiltering=FALSE)
rresABwo_AB_bran <- lfcShrink(G4_test_brandiff,contrast=c("Phase","ABwo","AB"),res=rresABwo_AB_bran,type="ashr")
rresABwo_AB_bran_sign <- rresABwo_AB_bran$padj<0.05
rresABwo_AB_bran_signOTU <- na.omit(colnames(G4_test_bran)[rresABwo_AB_bran_sign==TRUE])
rresABwo_AB_branvol <- volcanokim(rresABwo_AB_bran,taxlevel="species")+ggtitle("ABwo \u2013 AB")
mylegend_volcano_ABwoAB <- get_legend(rresABwo_AB_branvol)
rresABwo_AB_branvol <-rresABwo_AB_branvol+theme(legend.position="none")


# Concatenate all pairwise comparisons
##making dataframe with all sign ASVs + genera
G4_otunames_bran <- unique(c(resAB_Pre_bran_signOTU,rresABwo_AB_bran_signOTU))
G4_otunames_bran <- na.omit(G4_otunames_bran)
G4_phylumname_bran <- mapvalues(G4_otunames_bran,from=rownames(G4_Taxtable),to=as.character(G4_Taxtable$Phylum))
G4_phylumname_bran <- mapvalues(G4_phylumname_bran,from=G4_Taxtable$Genus,to=as.character(G4_Taxtable$Phylum))
G4_genusname <- mapvalues(G4_phylumname_bran,from=rownames(G4_Taxtable),to=as.character(G4_Taxtable$Genus))
G4_combinedgenus_phylum_bran <- data.frame(genus=G4_genusname,phylum=G4_phylumname_bran,otuname=G4_otunames_bran)

##adding p-values for all pairwise comparisons for each sign  ASV+genus + assigning letters
G4_saveotu_bran <- data.frame()
for (i in G4_otunames_bran){
pvalueAB_Pre <- resAB_Pre_bran[i,]$padj
names(pvalueAB_Pre) <- c("Prebiotic-ABwo")
pvalueABwo_AB <- rresABwo_AB_bran[i,]$padj
names(pvalueABwo_AB) <- c("AB-ABwo")
G4_pvaluetotal_bran <- c(pvalueAB_Pre,pvalueABwo_AB)
G4_letters_bran <- multcompView::multcompLetters(G4_pvaluetotal_bran)$"Letters"

#making dataframe with the counts of the sign ASVs/genera in all conditions + letters for annotating sign differences
G4_diffotu_bran <- plotCounts(G4_test_brandiff,gene=i,intgroup="Phase",returnData=TRUE)
G4_diffotu_bran <- cbind(G4_diffotu_bran,G4_testmetadata_bran)
G4_diffotu_bran <- mutate(G4_diffotu_bran,phylum=rep(G4_combinedgenus_phylum_bran[G4_combinedgenus_phylum_bran$otuname==i,]$phylum,nrow(G4_diffotu_bran)),genus=rep(G4_combinedgenus_phylum_bran[G4_combinedgenus_phylum_bran$otuname==i,]$genus,nrow(G4_diffotu_bran)),otu=i)
G4_diffotu_bran$labels<- mapvalues(G4_diffotu_bran$Phase,from=unique(G4_diffotu_bran$Phase),to=c(G4_letters_bran[3],G4_letters_bran[1],G4_letters_bran[2]))
G4_saveotu_bran <- rbind(G4_saveotu_bran,G4_diffotu_bran)
}

G4_saveotu_bran <- ddply(G4_saveotu_bran,.(Phase,otu),mutate, Q1=quantile(log(count),1/4), Q3=quantile(log(count),3/4),IQR=Q3-Q1,upper.limit=Q3+1.5*IQR,lower.limit=Q1-1.5*IQR)

# Order data in subpanels of multipanel graph
##Only working with genera
G4_otunames_bran_reorder <-  gtools::mixedsort(G4_otunames_bran[grep("ASV",G4_otunames_bran)])
G4_genus_names<-G4_otunames_bran[!G4_otunames_bran %in%  G4_otunames_bran[grep("ASV",G4_otunames_bran)]]

G4_saveotu_bran1<-G4_saveotu_bran[G4_saveotu_bran$otu%in%G4_genus_names[1:10],]
G4_saveotu_bran1<- G4_saveotu_bran1[gtools::mixedorder(G4_saveotu_bran1$otu),]
G4_saveotu_bran1$otu <- DescTools::reorder.factor(x=factor(G4_saveotu_bran1$otu))

G4_saveotu_bran2<-G4_saveotu_bran[G4_saveotu_bran$otu%in%G4_genus_names[11:20],]
G4_saveotu_bran2<- G4_saveotu_bran2[gtools::mixedorder(G4_saveotu_bran2$otu),]
G4_saveotu_bran2$otu <- DescTools::reorder.factor(x=factor(G4_saveotu_bran2$otu))


## Color strips by phylum level classification
G4_saveotu_branfillphylum <- rbind(G4_saveotu_bran1,G4_saveotu_bran2)
G4_saveotu_branfillphylum$fill <- mapvalues(G4_saveotu_branfillphylum$phylum,from=unique(G4_saveotu_branfillphylum$phylum),to=c("#953d6aB3","#cecb99","#5d6295B3","#837239B3","#557d7aB3"))
G4_saveotu_branfillphylum <-G4_saveotu_branfillphylum[!duplicated(G4_saveotu_branfillphylum$otu),]

#plotting to save the legend for future plots
plot_C10_bran <- ggplot(G4_saveotu_branfillphylum,aes(x=Phase,y=count,fill=G4_saveotu_branfillphylum$phylum))+ geom_boxplot() + scale_fill_manual(breaks=unique(G4_saveotu_branfillphylum$phylum),values=c("#953d6aB3","#cecb99","#5d6295B3","#837239B3","#557d7aB3"),name="Phylum")+theme(legend.position="bottom",legend.text=element_text(face='italic'))+guides(fill=guide_legend(nrow=3))
striplegend <- get_legend(plot_C10_bran)
G4_saveotu_branfillphylum1 <- G4_saveotu_branfillphylum[G4_saveotu_branfillphylum$otu%in%G4_saveotu_bran1$otu,]
G4_saveotu_branfillphylum2 <- G4_saveotu_branfillphylum[G4_saveotu_branfillphylum$otu%in%G4_saveotu_bran2$otu,]



## Create side-by-side box plots
plot_C10_bran <- ggplot(G4_saveotu_bran1,aes(x=Phase,y=count,fill=Phase))+ geom_boxplot()+geom_point(data=G4_saveotu_bran1[log(G4_saveotu_bran1$count)>G4_saveotu_bran1$upper.limit|log(G4_saveotu_bran1$count)<G4_saveotu_bran1$lower.limit,],aes(x=Phase,y=count,color=Phase),show.legend=FALSE)
plot_C10_bran <- plot_C10_bran +scale_y_continuous(labels=scientific,trans='log10',breaks = trans_breaks('log10', function(x) 10^x))+ylab("Normalised log transformed counts") +facet_grid(.~otu,drop=TRUE,scales='free')
plot_C10_bran <- papertheme(plot_C10_bran)+theme(axis.title.x = element_blank()) + scale_fill_manual(values=point_colors,name=NULL)+ scale_color_manual(values=point_colors,name=NULL)+ theme(legend.justification = "center") +theme(strip.text.x=element_text(size=10, face = "bold"),axis.text.x = element_blank(),axis.title.y = element_text(color="white"))+geom_text(aes(label=labels,x=Phase,y=max(count)+10))
deseqsaveotu1_bran <-plot_C10_bran+ theme(legend.position = "none")
deseqsaveotu1strips_bran<- ggplot_gtable(ggplot_build(deseqsaveotu1_bran))
stripr <- which(grepl('strip-t',deseqsaveotu1strips_bran$layout$name))
fills <- as.character(G4_saveotu_branfillphylum1$fill)
k <- 1
for (i in stripr) {
  j <- which(grepl('rect', deseqsaveotu1strips_bran$grobs[[i]]$grobs[[1]]$childrenOrder))
  deseqsaveotu1strips_bran$grobs[[i]]$grobs[[1]]$children[[j]]$gp$fill <- fills[k]
  k <- k+1
}

plot_C10_bran <- ggplot(G4_saveotu_bran2,aes(x=Phase,y=count,fill=Phase))+ geom_boxplot()+geom_point(data=G4_saveotu_bran2[log(G4_saveotu_bran2$count)>G4_saveotu_bran2$upper.limit|log(G4_saveotu_bran2$count)<G4_saveotu_bran2$lower.limit,],aes(x=Phase,y=count,color=Phase),show.legend=FALSE)
plot_C10_bran <- plot_C10_bran +scale_y_continuous(labels=scientific,trans='log10',breaks = trans_breaks('log10', function(x) 10^x))+ylab("Normalised log transformed counts") +facet_grid(.~otu,drop=TRUE,scales='free')
plot_C10_bran <- papertheme(plot_C10_bran)+theme(axis.title.x = element_blank()) + scale_fill_manual(values=point_colors,name=NULL)+ scale_color_manual(values=point_colors,name=NULL)+ theme(legend.justification = "center") +theme(strip.text.x=element_text(size=10, face = "bold"),axis.text.x = element_blank(),axis.title.y = element_text(color="white"))+geom_text(aes(label=labels,x=Phase,y=max(count)+10))
deseqGsaveotu2_bran <-plot_C10_bran+ theme(legend.position = "none")
deseqsaveotu2strips_bran <- ggplot_gtable(ggplot_build(deseqGsaveotu2_bran))
stripr <- which(grepl('strip-t',deseqsaveotu2strips_bran$layout$name))
fills <- as.character(G4_saveotu_branfillphylum2$fill)
k <- 1
for (i in stripr) {
  j <- which(grepl('rect', deseqsaveotu2strips_bran$grobs[[i]]$grobs[[1]]$childrenOrder))
  deseqsaveotu2strips_bran$grobs[[i]]$grobs[[1]]$children[[j]]$gp$fill <- fills[k]
  k <- k+1
}


##combining all plots and writing out to tiff file in output path
tiff(paste0(output_path,"DESeq_Genus.tiff"),width=12*dpi,height=16*dpi,res=dpi,compression="lzw")
p_C10_bran<-ggdraw(plot_grid(plot_grid(deseqsaveotu1strips_bran,deseqsaveotu2strips_bran,ncol=1,rel_heights = c(0.25,0.25,0.25,0.25)),plot_grid(mylegenddeseqpre,striplegend,ncol=2,rel_widths=c(0.5,0.5)),ncol=1,rel_heights=c(0.9,0.1)))+draw_label("Normalised log transformed counts",0.012,0.55,angle=90,size=20)
dev.off()
p_C10_bran
#ggsave(p_C10_bran, path = output_path, filename = "Bran_box", device = "png", width = 18, height = 12, limitsize=FALSE)
```

{D.1}


In [ ]:
```{r D.1 CIM}
G4_ASVtax_noNA <- subset(G4_ASVtax[,-c(1:5)], Genus!='NA')
G4_ASVtax_noNA <- rownames_to_column(G4_ASVtax_noNA)
G4_ASVtax_noNA[,1] <- paste(G4_ASVtax_noNA[,1], G4_ASVtax_noNA[,2], sep=" ")
rownames(G4_ASVtax_noNA) <- G4_ASVtax_noNA[,1]
G4_ASVtax_noNA <- G4_ASVtax_noNA[,-c(1:2)]

propionate_noNA <- subset(noNA, propionate!='NA')
G4_datatest <- merge(propionate_noNA,t(G4_ASVtax_noNA),by.x="row.names",by.y="row.names",all.x=FALSE,all.y=FALSE)
rownames(G4_datatest) <- propionate_noNA$Samplenumber

G4_testsampleinfo <- G4_datatest[,c(1:ncol(propionate_noNA))]
G4_datatest <- G4_datatest[,-c(1:ncol(propionate_noNA)+1)]
G4_datatest <- G4_datatest[,-1]
X <- low.count.removal(as.matrix(G4_datatest+1), percent=0.01)$data.filter
X <- logratio.transfo(X, logratio = 'CLR', offset = 0)

# create dataset with response
Y <- data.frame(G4_testsampleinfo$propionate, G4_testsampleinfo$acetate)
colnames(Y) <- c('propionate','acetate')

# choosing the number of components
propotu.pls <- pls(X, Y, ncomp = 10, mode = "regression")
propotu.spls <- spls(X, Y, ncomp = 10, mode = "regression")

Q.pls <- perf(propotu.pls, validation = "Mfold", folds = 10, progressBar = FALSE, nrepeat = 100)
Q.spls <- perf(propotu.spls, validation = "Mfold", folds = 10, progressBar = FALSE, nrepeat = 100)

# Q2 criterium
plot(Q.pls,criterion="Q2.total")
plot(Q.spls,criterion="Q2.total")

# Tune X components
#ONLY for spls
# set range of test values for number of variables to use from X dataframe
list.keepX <- c(seq(10,30,5))
list.keepY <- c(2,2)
set.seed(250)
tune.spls.propotu <- tune.spls(X+0.0000000000000001, Y+0.0000000000000001, ncomp = 2,
                              test.keepX = list.keepX,
                              test.keepY = list.keepY,
                              nrepeat = 1, folds = 10, # use 10 folds
                              mode = 'regression', measure = 'cor')
plot(tune.spls.propotu) # use the correlation measure for tuning
tune.spls.propotu$choice.keepX
tune.spls.propotu$choice.keepY

# FINAL model: 2 components will be fine, with 20 and 25 variables in sPLS
propotu.pls <- pls(X, Y, ncomp=2, mode = "regression")
propotu.spls <- spls(X, Y, ncomp=2, keepX=c(50,20), keepY=c(2,2), mode = "regression")

#OUTPUT
propotu.pls$prop_expl_var
propotu.spls$prop_expl_var

#Ordination
## ordination X space
plotIndiv(propotu.spls, comp = 1:2, rep.space= 'X-variate', group = G4_testsampleinfo$Donor,legend = TRUE, title = 'sPLS comp 1 - 2, X-space')
plotIndiv(propotu.pls, comp = 1:2, rep.space= 'X-variate', group = G4_testsampleinfo$Donor,legend = TRUE, title = 'PLS comp 1 - 2, X-space')

##ordination  Y space
plotIndiv(propotu.spls, comp = 1:2, rep.space= 'Y-variate', group = G4_testsampleinfo$Donor,legend = TRUE, title = 'sPLS comp 1 - 2, Y-space')
plotIndiv(propotu.pls, comp = 1:2, rep.space= 'Y-variate', group = G4_testsampleinfo$Donor,legend = TRUE, title = 'PLS comp 1 - 2, Y-space')

# correlation circle
plotVar(propotu.pls, comp = 1:2, cex = c(5, 5))
plotVar(propotu.spls, comp = 1:2, cex = c(5, 5))

#clustered image map
cimob <- cim(propotu.spls, comp = 1:2, xlab = "Functional response variables", ylab = "ASV Genus", margins = c(7,50),mapping="XY",color=rev(color.spectral(50)))
cimob$mat

tiff(paste0(output_path,"cim_genus.tiff"),width=12*dpi,height=6*dpi,res=dpi,compression="lzw")
dev.off()
```